# SWORD River Trace

Displays SWORD reaches that intersect a study reach boundary, then traces
the full mainstem river upstream and downstream from a user-selected
SWORD reach ID.

## Setup

1. Set the paths in the **CONFIG** cell below.
2. Run the display cell to see which SWORD reaches overlap your study reach.
3. Choose a `reach_id` from the map/table and enter it in the trace cell.
4. Run the trace cell to extract and display the full mainstem.

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

CONFIG = {
    "reach_shapefile": Path(
        r"E:\Dissertation\Data\RiverMapping\Reaches\Jurua_SantosDumont\Jurua_SantosDumont.shp"
    ),
    "sword_dir": Path(r"E:\Dissertation\Data\SWORD"),
    "sword_version": "v17b",
    "sword_cont_abb": "sa",  # af, as, eu, na, oc, sa
}

sword_reaches_path = (
    CONFIG["sword_dir"]
    / f"{CONFIG['sword_cont_abb']}_sword_reaches_{CONFIG['sword_version']}.gpkg"
)

print(f"Reach shapefile:  {CONFIG['reach_shapefile']}")
print(f"SWORD reaches:    {sword_reaches_path}")

In [ ]:
# Load study reach boundary
reach_boundary = gpd.read_file(CONFIG["reach_shapefile"])

# Load SWORD reaches for the continent
print(f"Loading {sword_reaches_path.name} ...")
sword_reaches = gpd.read_file(sword_reaches_path, engine="pyogrio")
print(f"Loaded {len(sword_reaches):,} SWORD reaches.")

# Reproject reach boundary to match SWORD CRS if needed
if reach_boundary.crs != sword_reaches.crs:
    reach_boundary = reach_boundary.to_crs(sword_reaches.crs)

# Find SWORD reaches that intersect the reach boundary
intersecting = gpd.sjoin(
    sword_reaches, reach_boundary, how="inner", predicate="intersects"
).drop(columns="index_right", errors="ignore")
intersecting = intersecting.drop_duplicates(subset="reach_id")

print(f"\n{len(intersecting)} SWORD reaches intersect the reach boundary:\n")
display_cols = [c for c in ["reach_id", "reach_len", "dist_out", "facc", "slope", "width", "type"]
                if c in intersecting.columns]
print(intersecting[display_cols].to_string(index=False))

# Build lookup dictionaries for tracing (used in the next cell)
dn_lookup = sword_reaches.set_index("reach_id")["rch_id_dn"].to_dict()
dist_lookup = sword_reaches.set_index("reach_id")["dist_out"].to_dict()
facc_lookup = sword_reaches.set_index("reach_id")["facc"].to_dict()

upstream_lookup = {}
for _, row in sword_reaches.iterrows():
    rid = int(row["reach_id"])
    dn_raw = str(row["rch_id_dn"])
    dn_ids = [int(x) for x in dn_raw.split() if int(x) != 0]
    for dn_id in dn_ids:
        upstream_lookup.setdefault(dn_id, []).append(rid)
print(f"\nUpstream lookup built for {len(upstream_lookup):,} reaches.")

# Plot
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
reach_boundary.plot(ax=ax, facecolor="lightblue", edgecolor="black",
                    linewidth=2, alpha=0.3, label="Reach boundary")
intersecting.plot(ax=ax, color="red", linewidth=2.5, label="SWORD reaches")
for _, row in intersecting.iterrows():
    centroid = row.geometry.centroid
    ax.annotate(
        str(int(row["reach_id"])),
        xy=(centroid.x, centroid.y),
        fontsize=7, ha="center", va="center",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="yellow",
                  edgecolor="gray", alpha=0.85),
    )
ax.set_title("Study reach with intersecting SWORD reaches")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend(loc="best")
plt.tight_layout()
plt.show()

## Select a SWORD reach

From the map and table above, choose a SWORD `reach_id` that falls within
your study reach. Enter it in the cell below, then run the cell to trace
the full mainstem river upstream and downstream.

In [ ]:
# ── USER INPUT ──────────────────────────────────────────────────────
SELECTED_REACH_ID = 62296301011  # <-- Replace with a reach_id from the map above
# ────────────────────────────────────────────────────────────────────

if SELECTED_REACH_ID not in dist_lookup:
    raise ValueError(
        f"Reach {SELECTED_REACH_ID} not found in SWORD "
        f"{CONFIG['sword_cont_abb']} dataset. Check the reach_id."
    )


def trace_upstream_mainstem(reach_id, max_iter=5000):
    """Walk upstream along the mainstem (largest facc at confluences)."""
    path = []
    current = reach_id
    for _ in range(max_iter):
        candidates = upstream_lookup.get(current, [])
        if not candidates:
            break
        best = max(candidates, key=lambda r: facc_lookup.get(r, 0))
        path.append(best)
        current = best
    return list(reversed(path))


def trace_downstream(reach_id, max_iter=5000):
    """Walk downstream via rch_id_dn (first non-zero entry at splits)."""
    path = []
    current = reach_id
    for _ in range(max_iter):
        dn_raw = dn_lookup.get(current)
        if dn_raw is None:
            break
        dn_ids = [int(x) for x in str(dn_raw).split() if int(x) != 0]
        if not dn_ids:
            break
        current = dn_ids[0]
        path.append(current)
    return path


upstream_path = trace_upstream_mainstem(SELECTED_REACH_ID)
downstream_path = trace_downstream(SELECTED_REACH_ID)
full_path = upstream_path + [SELECTED_REACH_ID] + downstream_path

print(f"Upstream reaches:   {len(upstream_path)}")
print(f"Downstream reaches: {len(downstream_path)}")
print(f"Total reaches:      {len(full_path)}")

# Extract and order the traced reaches
river_reaches = sword_reaches[sword_reaches["reach_id"].isin(full_path)].copy()
order_map = {rid: i + 1 for i, rid in enumerate(full_path)}
river_reaches["reach_order"] = river_reaches["reach_id"].map(order_map)
river_reaches = river_reaches.sort_values("reach_order").reset_index(drop=True)

selected_gdf = river_reaches[river_reaches["reach_id"] == SELECTED_REACH_ID]

# Plot full river
fig, ax = plt.subplots(1, 1, figsize=(14, 10))
river_reaches.plot(
    ax=ax, column="reach_order", cmap="viridis", linewidth=2,
    legend=True, legend_kwds={"label": "Reach order (US to DS)", "shrink": 0.6},
)
selected_gdf.plot(ax=ax, color="red", linewidth=4, label="Selected reach")
reach_boundary.plot(
    ax=ax, facecolor="none", edgecolor="red", linewidth=2,
    linestyle="--", label="Study reach boundary",
)
ax.set_title(
    f"SWORD mainstem trace: {len(full_path)} reaches\n"
    f"Selected reach: {SELECTED_REACH_ID}"
)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend(loc="best")
plt.tight_layout()
plt.show()

## Longitudinal Profile

Sample DEM elevation at the midpoint of each traced SWORD reach via
Google Earth Engine (NASADEM < 60° lat, ArcticDEM ≥ 60°), then plot
a longitudinal elevation profile from headwaters to mouth.

In [ ]:
import ee
import time
import numpy as np

# Initialise GEE — add project="your-project-id" if your setup requires it
ee.Initialize()

ARCTICDEM_MIN_LATITUDE = 60.0


def get_elevation(lat, lon, max_retries=3):
    """Sample DEM elevation at (lat, lon) via Google Earth Engine."""
    for attempt in range(max_retries):
        try:
            point = ee.Geometry.Point([lon, lat])
            if lat >= ARCTICDEM_MIN_LATITUDE:
                dem = ee.Image("UMN/PGC/ArcticDEM/V3/2m_mosaic")
                scale = 32
            else:
                dem = ee.Image("NASA/NASADEM_HGT/001")
                scale = 30
            sample = dem.select("elevation").sample(point, scale).first()
            if sample is None:
                return None
            elev = sample.get("elevation").getInfo()
            if elev is None or not (-500 < elev < 9000):
                return None
            return float(elev)
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
                continue
            print(f"  Failed at ({lat:.4f}, {lon:.4f}): {e}")
            return None
    return None


# Sample one elevation at the midpoint of each traced SWORD reach
print(f"Sampling elevation for {len(river_reaches)} reaches ...\n")
elevations = []
for _, row in river_reaches.iterrows():
    midpoint = row.geometry.interpolate(0.5, normalized=True)
    elev = get_elevation(midpoint.y, midpoint.x)
    elevations.append({
        "reach_id": int(row["reach_id"]),
        "reach_order": int(row["reach_order"]),
        "dist_out": float(row["dist_out"]),
        "latitude": midpoint.y,
        "longitude": midpoint.x,
        "elevation_m": elev,
    })
    status = f"{elev:.1f} m" if elev is not None else "FAILED"
    print(f"  {int(row['reach_order']):3d}/{len(river_reaches)}  "
          f"reach_id={int(row['reach_id'])}  "
          f"dist_out={row['dist_out']/1000:.0f} km  "
          f"elev={status}")

elev_df = pd.DataFrame(elevations)
valid = elev_df["elevation_m"].notna().sum()
print(f"\nSampled {valid}/{len(elev_df)} reaches successfully.")

In [ ]:
valid_df = elev_df.dropna(subset=["elevation_m"]).copy()

# Distance from headwaters: 0 = most upstream, increasing downstream
max_dist_out = valid_df["dist_out"].max()
valid_df["dist_from_us_km"] = (max_dist_out - valid_df["dist_out"]) / 1000

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(
    valid_df["dist_from_us_km"], valid_df["elevation_m"],
    "o-", color="steelblue", markersize=4, linewidth=1.5,
    label="SWORD reaches",
)

# Highlight selected reach
selected_elev = valid_df[valid_df["reach_id"] == SELECTED_REACH_ID]
if not selected_elev.empty:
    ax.plot(
        selected_elev["dist_from_us_km"], selected_elev["elevation_m"],
        "ro", markersize=10, zorder=5, label="Selected reach",
    )

ax.set_title(
    f"Longitudinal profile: {len(valid_df)} SWORD reaches\n"
    f"Selected reach: {SELECTED_REACH_ID}"
)
ax.set_xlabel("Distance from headwaters (km)")
ax.set_ylabel("Elevation (m)")
ax.legend(loc="best")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Elevation range: {valid_df['elevation_m'].min():.1f} – "
      f"{valid_df['elevation_m'].max():.1f} m")
print(f"River length:    {valid_df['dist_from_us_km'].max():.1f} km")

In [ ]:
import contextily as ctx
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

# Reproject to Web Mercator so Esri satellite tiles align cleanly
river_3857    = river_reaches.to_crs(epsg=3857)
selected_3857 = river_3857[river_3857["reach_id"] == SELECTED_REACH_ID]
boundary_3857 = reach_boundary.to_crs(epsg=3857)

fig = plt.figure(figsize=(16, 13))
gs = fig.add_gridspec(2, 2, height_ratios=[3, 1], width_ratios=[2, 1])
ax_map  = fig.add_subplot(gs[0, 0])   # detailed mainstem
ax_ctx  = fig.add_subplot(gs[0, 1])   # zoomed-out context
ax_prof = fig.add_subplot(gs[1, :])   # long profile, full width

# ── Top-left: mainstem over satellite basemap ───────────────────────
river_3857.plot(
    ax=ax_map, column="reach_order", cmap="viridis", linewidth=2.5,
    legend=True,
    legend_kwds={"label": "Reach order (US \u2192 DS)", "shrink": 0.5},
)
selected_3857.plot(ax=ax_map, color="red", linewidth=4)
boundary_3857.plot(
    ax=ax_map, facecolor="none", edgecolor="red",
    linewidth=2, linestyle="--",
)
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

ax_map.legend(
    handles=[
        Line2D([0], [0], color="red", lw=4, label="Selected reach"),
        Line2D([0], [0], color="red", lw=2, ls="--",
               label="Study reach boundary"),
    ],
    loc="best",
)
ax_map.set_title(
    f"SWORD mainstem trace ({len(full_path)} reaches)\n"
    f"Selected reach: {SELECTED_REACH_ID}"
)
ax_map.set_xlabel("Easting (m, Web Mercator)")
ax_map.set_ylabel("Northing (m, Web Mercator)")

# ── Top-right: zoomed-out context (~1:4 of the main map extent) ──────
# Capture the main map's final extent, then expand ×4 about its center
xmin, xmax = ax_map.get_xlim()
ymin, ymax = ax_map.get_ylim()
xc, yc = (xmin + xmax) / 2, (ymin + ymax) / 2
half_w = (xmax - xmin) / 2 * 4
half_h = (ymax - ymin) / 2 * 4

river_3857.plot(ax=ax_ctx, color="gold", linewidth=1.2)
selected_3857.plot(ax=ax_ctx, color="red", linewidth=2.5)

# Set the expanded extent BEFORE add_basemap so tiles cover the wider view.
# add_basemap's default reset_extent=True then preserves these limits.
ax_ctx.set_xlim(xc - half_w, xc + half_w)
ax_ctx.set_ylim(yc - half_h, yc + half_h)
ax_ctx.set_aspect("equal")
ctx.add_basemap(ax_ctx, source=ctx.providers.Esri.WorldImagery)

# Box showing the main map's footprint
ax_ctx.add_patch(Rectangle(
    (xmin, ymin), xmax - xmin, ymax - ymin,
    fill=False, edgecolor="yellow", linewidth=1.5,
))
ax_ctx.set_title("Regional context (\u22481:4)")
ax_ctx.set_xticks([])
ax_ctx.set_yticks([])

# ── Bottom: longitudinal profile ────────────────────────────────────
ax_prof.plot(
    valid_df["dist_from_us_km"], valid_df["elevation_m"],
    "o-", color="steelblue", markersize=4, linewidth=1.5,
    label="SWORD reaches",
)
selected_elev = valid_df[valid_df["reach_id"] == SELECTED_REACH_ID]
if not selected_elev.empty:
    ax_prof.plot(
        selected_elev["dist_from_us_km"], selected_elev["elevation_m"],
        "ro", markersize=10, zorder=5, label="Selected reach",
    )
ax_prof.set_title("Longitudinal profile")
ax_prof.set_xlabel("Distance from headwaters (km)")
ax_prof.set_ylabel("Elevation (m)")
ax_prof.legend(loc="best")
ax_prof.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
"""Calculate fraction of distance downstream and drainage area for river reaches.

Uses SWORD v17b topology to trace the full mainstem river upstream and
downstream from each study reach, then computes normalised position
metrics.  Elevation is sampled at each SWORD reach midpoint via GEE
(NASADEM / ArcticDEM) to build a longitudinal profile.

Usage
-----
    python calc_fraction_ds.py <master_csv> <sword_cont_abb> [--ee-project ID]

Arguments
---------
    master_csv       Path to master CSV.  Required columns:
                         river_name, working_directory, SWORD_cont_abb
    sword_cont_abb   Continent to process this run (af, as, eu, na, oc, sa).
                     Only rivers whose SWORD_cont_abb matches are processed.

Outputs
-------
    {OUTPUT_DIR}/{river_name}/{river_name}_SWORD_reaches.gpkg
    {OUTPUT_DIR}/{cont}_fraction_ds_results.csv
    {OUTPUT_DIR}/{cont}_fraction_ds_report.pdf
"""

from __future__ import annotations

import argparse
import os
import sys
import time

import geopandas as gpd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd

try:
    import contextily as ctx
    HAS_CTX = True
except ImportError:
    HAS_CTX = False

import ee

# ── Constants ──────────────────────────────────────────────────────
SWORD_DIR = r"E:\Dissertation\Data\SWORD"
OUTPUT_DIR = (
    r"E:\Dissertation\Data\SWORD\Extracted_Rivers\Chapter_2_dist_ds_analysis"
)
master_csv_path = r"E:\Dissertation\Data\GreenbergZhaoRees_river_datasheet.csv"
SWORD_VERSION = "v17b"
ARCTICDEM_MIN_LATITUDE = 60.0
VALID_CONTINENTS = {"sa"}


# ── Elevation sampling via GEE ─────────────────────────────────────

def get_elevation(lat: float, lon: float, max_retries: int = 3) -> float | None:
    """Sample DEM elevation at (lat, lon) via Google Earth Engine."""
    for attempt in range(max_retries):
        try:
            point = ee.Geometry.Point([lon, lat])
            if lat >= ARCTICDEM_MIN_LATITUDE:
                dem = ee.Image("UMN/PGC/ArcticDEM/V3/2m_mosaic")
                scale = 32
            else:
                dem = ee.Image("NASA/NASADEM_HGT/001")
                scale = 30
            sample = dem.select("elevation").sample(point, scale).first()
            if sample is None:
                return None
            elev = sample.get("elevation").getInfo()
            if elev is None or not (-500 < elev < 9000):
                return None
            return float(elev)
        except Exception:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
                continue
            return None
    return None


def sample_river_elevations(
    river_reaches: gpd.GeoDataFrame,
) -> pd.DataFrame:
    """Sample elevation at the midpoint of each traced SWORD reach."""
    records: list[dict] = []
    n = len(river_reaches)
    for i, (_, row) in enumerate(river_reaches.iterrows()):
        midpoint = row.geometry.interpolate(0.5, normalized=True)
        elev = get_elevation(midpoint.y, midpoint.x)
        records.append({
            "reach_id": int(row["reach_id"]),
            "reach_order": int(row["reach_order"]),
            "dist_out": float(row["dist_out"]),
            "elevation_m": elev,
        })
        if (i + 1) % 25 == 0 or i == n - 1:
            ok = sum(1 for r in records if r["elevation_m"] is not None)
            print(f"      ... {i + 1}/{n} sampled ({ok} OK)")
    return pd.DataFrame(records)


# ── SWORD topology helpers ─────────────────────────────────────────

def build_lookups(sword_reaches: gpd.GeoDataFrame) -> tuple[dict, dict, dict, dict]:
    """Return (dn_lookup, dist_lookup, facc_lookup, upstream_lookup) from SWORD reaches."""
    dn_lookup = sword_reaches.set_index("reach_id")["rch_id_dn"].to_dict()
    dist_lookup = sword_reaches.set_index("reach_id")["dist_out"].to_dict()
    facc_lookup = sword_reaches.set_index("reach_id")["facc"].to_dict()

    upstream_lookup: dict[int, list[int]] = {}
    for _, row in sword_reaches.iterrows():
        rid = int(row["reach_id"])
        dn_raw = str(row["rch_id_dn"])
        dn_ids = [int(x) for x in dn_raw.split() if int(x) != 0]
        for dn_id in dn_ids:
            upstream_lookup.setdefault(dn_id, []).append(rid)

    return dn_lookup, dist_lookup, facc_lookup, upstream_lookup


def trace_upstream_mainstem(
    reach_id: int,
    upstream_lookup: dict,
    facc_lookup: dict,
    max_iter: int = 5000,
) -> list[int]:
    """Walk upstream along the mainstem (largest facc at confluences)."""
    path: list[int] = []
    current = reach_id
    for _ in range(max_iter):
        candidates = upstream_lookup.get(current, [])
        if not candidates:
            break
        best = max(candidates, key=lambda r: facc_lookup.get(r, 0))
        path.append(best)
        current = best
    return list(reversed(path))


def trace_downstream(
    reach_id: int,
    dn_lookup: dict,
    max_iter: int = 5000,
) -> list[int]:
    """Walk downstream via rch_id_dn (first non-zero entry at splits)."""
    path: list[int] = []
    current = reach_id
    for _ in range(max_iter):
        dn_raw = dn_lookup.get(current)
        if dn_raw is None:
            break
        dn_ids = [int(x) for x in str(dn_raw).split() if int(x) != 0]
        if not dn_ids:
            break
        current = dn_ids[0]
        path.append(current)
    return path


# ── PDF page ───────────────────────────────────────────────────────

def _add_basemap(ax, **kwargs):
    """Attempt to add an Esri satellite basemap; silently skip on failure."""
    if not HAS_CTX:
        return
    try:
        ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery, **kwargs)
    except Exception:
        pass


def draw_page(
    pdf: PdfPages,
    river_name: str,
    ds_order: int,
    river_3857: gpd.GeoDataFrame,
    ds_boundary_3857: gpd.GeoDataFrame,
    elev_df: pd.DataFrame,
    sword_reach_id: int | None,
    frac_dist_ds: float | None,
    frac_dr_ar: float | None,
) -> None:
    """Draw one three-panel page in the report PDF."""
    fig = plt.figure(figsize=(16, 13))
    gs = fig.add_gridspec(2, 2, height_ratios=[3, 1], width_ratios=[2, 1])
    ax_map = fig.add_subplot(gs[0, 0])
    ax_ctx = fig.add_subplot(gs[0, 1])
    ax_prof = fig.add_subplot(gs[1, :])

    # Highlight the SWORD reach for this ds_order
    selected_3857 = (
        river_3857[river_3857["reach_id"] == sword_reach_id]
        if sword_reach_id is not None
        else river_3857.iloc[:0]
    )

    # ── Top-left: mainstem over satellite ──────────────────────────
    river_3857.plot(
        ax=ax_map, column="reach_order", cmap="viridis", linewidth=2.5,
        legend=True,
        legend_kwds={"label": "Reach order (US \u2192 DS)", "shrink": 0.5},
    )
    if not selected_3857.empty:
        selected_3857.plot(ax=ax_map, color="red", linewidth=4)
    ds_boundary_3857.plot(
        ax=ax_map, facecolor="none", edgecolor="red",
        linewidth=2, linestyle="--",
    )
    _add_basemap(ax_map)

    legend_handles = [
        Line2D([0], [0], color="red", lw=2, ls="--",
               label=f"Reach {ds_order} boundary"),
    ]
    if sword_reach_id is not None:
        legend_handles.insert(
            0, Line2D([0], [0], color="red", lw=4,
                      label=f"SWORD {sword_reach_id}"),
        )
    ax_map.legend(handles=legend_handles, loc="best")

    if frac_dist_ds is not None and frac_dr_ar is not None:
        title = (
            f"{river_name} \u2014 Reach {ds_order}\n"
            f"frac_dist_ds = {frac_dist_ds:.4f}   "
            f"frac_dr_ar = {frac_dr_ar:.4f}"
        )
    else:
        title = f"{river_name} \u2014 Reach {ds_order}\n(metrics unavailable)"
    ax_map.set_title(title)
    ax_map.set_xlabel("Easting (m)")
    ax_map.set_ylabel("Northing (m)")

    # ── Top-right: zoomed-out context ──────────────────────────────
    xmin, xmax = ax_map.get_xlim()
    ymin, ymax = ax_map.get_ylim()
    xc, yc = (xmin + xmax) / 2, (ymin + ymax) / 2
    hw = (xmax - xmin) * 2
    hh = (ymax - ymin) * 2

    river_3857.plot(ax=ax_ctx, color="gold", linewidth=1.2)
    if not selected_3857.empty:
        selected_3857.plot(ax=ax_ctx, color="red", linewidth=2.5)

    ax_ctx.set_xlim(xc - hw, xc + hw)
    ax_ctx.set_ylim(yc - hh, yc + hh)
    ax_ctx.set_aspect("equal")
    _add_basemap(ax_ctx)

    ax_ctx.add_patch(Rectangle(
        (xmin, ymin), xmax - xmin, ymax - ymin,
        fill=False, edgecolor="yellow", linewidth=1.5,
    ))
    ax_ctx.set_title("Regional context")
    ax_ctx.set_xticks([])
    ax_ctx.set_yticks([])

    # ── Bottom: longitudinal profile ───────────────────────────────
    valid = elev_df.dropna(subset=["elevation_m"]).copy()
    if not valid.empty:
        max_do = valid["dist_out"].max()
        valid["dist_km"] = (max_do - valid["dist_out"]) / 1000

        ax_prof.plot(
            valid["dist_km"], valid["elevation_m"],
            "o-", color="steelblue", markersize=3, linewidth=1.2,
            label="SWORD reaches",
        )
        if sword_reach_id is not None:
            sel = valid[valid["reach_id"] == sword_reach_id]
            if not sel.empty:
                ax_prof.plot(
                    sel["dist_km"], sel["elevation_m"],
                    "ro", markersize=10, zorder=5,
                    label=f"Reach {ds_order}",
                )
        ax_prof.legend(loc="best")

    ax_prof.set_title("Longitudinal profile")
    ax_prof.set_xlabel("Distance from headwaters (km)")
    ax_prof.set_ylabel("Elevation (m)")
    ax_prof.grid(True, alpha=0.3)

    plt.tight_layout()
    pdf.savefig(fig)
    plt.close(fig)


# ── Per-river processing ──────────────────────────────────────────

def process_river(
    river_name: str,
    working_directory: str,
    sword_reaches: gpd.GeoDataFrame,
    dn_lookup: dict,
    dist_lookup: dict,
    facc_lookup: dict,
    upstream_lookup: dict,
) -> tuple[
    gpd.GeoDataFrame | None,
    gpd.GeoDataFrame | None,
    pd.DataFrame | None,
    list[dict] | None,
]:
    """Trace, sample, and compute metrics for one river.

    Returns (river_reaches, reach_gdf, elev_df, result_rows) or four Nones.
    """
    # ── Load reach shapefile ───────────────────────────────────────
    reach_path = os.path.join(
        working_directory, "RiverMapping", "Reaches",
        river_name, f"{river_name}.shp",
    )
    if not os.path.isfile(reach_path):
        print(f"    ERROR: reach shapefile not found: {reach_path}")
        return None, None, None, None

    reach_gdf = gpd.read_file(reach_path)
    if reach_gdf.crs != sword_reaches.crs:
        reach_gdf = reach_gdf.to_crs(sword_reaches.crs)

    # ── Find SWORD reaches inside the study area ───────────────────
    intersecting = gpd.sjoin(
        sword_reaches, reach_gdf, how="inner", predicate="intersects",
    ).drop(columns="index_right", errors="ignore")
    intersecting = intersecting.drop_duplicates(subset="reach_id")

    if intersecting.empty:
        print("    ERROR: no SWORD reaches intersect the reach boundary")
        return None, None, None, None

    # Auto-identify anchor: most downstream SWORD reach in the area
    anchor_idx = intersecting["dist_out"].idxmin()
    anchor_id = int(intersecting.loc[anchor_idx, "reach_id"])
    print(
        f"    Anchor SWORD reach: {anchor_id}  "
        f"(dist_out = {intersecting.loc[anchor_idx, 'dist_out'] / 1000:.0f} km)"
    )

    # ── Trace full mainstem ────────────────────────────────────────
    us_path = trace_upstream_mainstem(anchor_id, upstream_lookup, facc_lookup)
    ds_path = trace_downstream(anchor_id, dn_lookup)
    full_path = us_path + [anchor_id] + ds_path
    print(
        f"    Traced: {len(us_path)} US + 1 + {len(ds_path)} DS "
        f"= {len(full_path)} reaches"
    )

    river_reaches = sword_reaches[
        sword_reaches["reach_id"].isin(full_path)
    ].copy()
    order_map = {rid: i + 1 for i, rid in enumerate(full_path)}
    river_reaches["reach_order"] = river_reaches["reach_id"].map(order_map)
    river_reaches = river_reaches.sort_values("reach_order").reset_index(
        drop=True
    )

    # ── Save .gpkg ─────────────────────────────────────────────────
    river_dir = os.path.join(OUTPUT_DIR, river_name)
    os.makedirs(river_dir, exist_ok=True)
    gpkg_path = os.path.join(river_dir, f"{river_name}_SWORD_reaches.gpkg")
    river_reaches.to_file(gpkg_path, driver="GPKG")
    print(f"    Saved: {gpkg_path}")

    # ── Sample elevations ──────────────────────────────────────────
    print(f"    Sampling elevations ({len(river_reaches)} reaches) ...")
    elev_df = sample_river_elevations(river_reaches)
    ok = elev_df["elevation_m"].notna().sum()
    print(f"    Elevation complete: {ok}/{len(elev_df)} successful")

    # ── Compute per-reach metrics ──────────────────────────────────
    max_dist_out = river_reaches["dist_out"].max()
    min_dist_out = river_reaches["dist_out"].min()
    total_dist = max_dist_out - min_dist_out

    # Use max facc across all traced reaches (handles distributary systems)
    has_facc = "facc" in river_reaches.columns
    if has_facc:
        max_facc = float(river_reaches["facc"].max())
    else:
        max_facc = None

    # One spatial join: map traced SWORD reaches → ds_order polygons
    reach_polys = reach_gdf[["ds_order", "geometry"]].copy()
    reach_polys = reach_polys.rename(columns={"ds_order": "ds_order_poly"})

    sword_to_ds = gpd.sjoin(
        river_reaches, reach_polys, how="inner", predicate="intersects",
    ).drop(columns="index_right", errors="ignore")

    results: list[dict] = []
    for ds_order in sorted(reach_gdf["ds_order"].unique()):
        ds_order = int(ds_order)
        matches = sword_to_ds[sword_to_ds["ds_order_poly"] == ds_order]

        if matches.empty:
            print(f"    ds_order {ds_order}: no SWORD overlap")
            results.append({
                "river_name": river_name,
                "ds_order": ds_order,
                "frac_dist_ds": None,
                "frac_dr_ar": None,
                "sword_reach_id": None,
            })
            continue

        target = matches.loc[matches["dist_out"].idxmin()]
        target_dist = float(target["dist_out"])
        target_id = int(target["reach_id"])

        frac_dist_ds = (
            (max_dist_out - target_dist) / total_dist
            if total_dist > 0 else None
        )
        frac_dr_ar = (
            float(target["facc"]) / max_facc
            if has_facc and max_facc and max_facc > 0 else None
        )

        results.append({
            "river_name": river_name,
            "ds_order": ds_order,
            "frac_dist_ds": frac_dist_ds,
            "frac_dr_ar": frac_dr_ar,
            "sword_reach_id": target_id,
        })

        metric_str = (
            f"frac_dist_ds={frac_dist_ds:.4f}  frac_dr_ar={frac_dr_ar:.4f}"
            if frac_dist_ds is not None and frac_dr_ar is not None
            else "metrics unavailable"
        )
        print(f"    ds_order {ds_order}: SWORD {target_id}  {metric_str}")

    return river_reaches, reach_gdf, elev_df, results


# ── Main driver ────────────────────────────────────────────────────

def run(
    master_csv_path: str,
    sword_cont_abb: str,
    ee_project: str | None = None,
) -> None:
    """Run fraction-downstream analysis for all rivers on one continent."""
    sword_cont_abb = sword_cont_abb.lower()
    if sword_cont_abb not in VALID_CONTINENTS:
        raise ValueError(
            f"Invalid continent '{sword_cont_abb}'. "
            f"Must be one of {sorted(VALID_CONTINENTS)}."
        )

    # ── Load and filter master CSV ─────────────────────────────────
    master = pd.read_csv(master_csv_path)
    required = {"river_name", "working_directory", "SWORD_cont_abb"}
    missing = required - set(master.columns)
    if missing:
        raise ValueError(f"Master CSV missing columns: {missing}")

    master["SWORD_cont_abb"] = master["SWORD_cont_abb"].str.strip().str.lower()
    rivers = master[master["SWORD_cont_abb"] == sword_cont_abb].reset_index(
        drop=True
    )
    if rivers.empty:
        print(f"No rivers found for continent '{sword_cont_abb}'.")
        return

    print(f"Rivers to process ({sword_cont_abb.upper()}): {len(rivers)}")
    for _, r in rivers.iterrows():
        print(f"  - {r['river_name']}")
    print()

    # ── Load SWORD (once per continent) ────────────────────────────
    sword_path = os.path.join(
        SWORD_DIR,
        f"{sword_cont_abb}_sword_reaches_{SWORD_VERSION}.gpkg",
    )
    print(f"Loading {os.path.basename(sword_path)} ...")
    sword_reaches = gpd.read_file(sword_path, engine="pyogrio")
    print(f"Loaded {len(sword_reaches):,} SWORD reaches.")

    if "facc" not in sword_reaches.columns:
        print(
            "WARNING: 'facc' column not found in SWORD reaches. "
            "frac_dr_ar will be unavailable."
        )
    print()

    # ── Build topology lookups (once) ──────────────────────────────
    print("Building topology lookups ...")
    dn_lookup, dist_lookup, facc_lookup, upstream_lookup = build_lookups(sword_reaches)
    print(f"Upstream lookup: {len(upstream_lookup):,} entries.\n")

    # ── Initialise GEE (once) ──────────────────────────────────────
    print("Initialising Google Earth Engine ...")
    if ee_project:
        ee.Initialize(project=ee_project)
    else:
        ee.Initialize()
    print("GEE ready.\n")

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # ── Process each river ─────────────────────────────────────────
    all_results: list[dict] = []
    pdf_queue: list[tuple] = []

    for i, (_, row) in enumerate(rivers.iterrows()):
        river_name = row["river_name"]
        working_dir = row["working_directory"]

        print("=" * 60)
        print(f"[{i + 1}/{len(rivers)}]  {river_name}")
        print("=" * 60)

        river_reaches, reach_gdf, elev_df, results = process_river(
            river_name, working_dir, sword_reaches,
            dn_lookup, dist_lookup, facc_lookup, upstream_lookup,
        )

        if results is not None:
            all_results.extend(results)
            pdf_queue.append(
                (river_name, river_reaches, reach_gdf, elev_df, results)
            )
        print()

    # ── Write master results CSV ───────────────────────────────────
    if all_results:
        results_df = pd.DataFrame(all_results)
        csv_out = os.path.join(
            OUTPUT_DIR, f"{sword_cont_abb}_fraction_ds_results.csv"
        )
        results_df.to_csv(csv_out, index=False)
        print(f"Results CSV: {csv_out}")
        print(results_df.to_string(index=False))
        print()

    # ── Generate PDF report ────────────────────────────────────────
    if pdf_queue:
        pdf_out = os.path.join(
            OUTPUT_DIR, f"{sword_cont_abb}_fraction_ds_report.pdf"
        )
        print(f"Generating PDF: {pdf_out}")

        with PdfPages(pdf_out) as pdf:
            for river_name, river_reaches, reach_gdf, elev_df, results in pdf_queue:
                river_3857 = river_reaches.to_crs(epsg=3857)
                reach_gdf_3857 = reach_gdf.to_crs(epsg=3857)

                for res in results:
                    ds_order = res["ds_order"]
                    sword_rid = res.get("sword_reach_id")

                    ds_bnd = reach_gdf_3857[
                        reach_gdf_3857["ds_order"] == ds_order
                    ]
                    if ds_bnd.empty:
                        continue

                    draw_page(
                        pdf,
                        river_name,
                        ds_order,
                        river_3857,
                        ds_bnd,
                        elev_df,
                        sword_rid,
                        res.get("frac_dist_ds"),
                        res.get("frac_dr_ar"),
                    )
                    print(f"  Page: {river_name}  ds_order={ds_order}")

        print(f"\nPDF complete: {pdf_out}")

    print("\nDone.")


# ── CLI ────────────────────────────────────────────────────────────

if __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description=(
            "Calculate fraction of distance downstream (frac_dist_ds) "
            "and fraction of drainage area (frac_dr_ar) for each river "
            "reach using SWORD v17b topology."
        ),
    )
    parser.add_argument(
        "master_csv",
        help="Path to master CSV (river_name, working_directory, SWORD_cont_abb).",
    )
    parser.add_argument(
        "sword_cont_abb",
        help="SWORD continent abbreviation to process (af, as, eu, na, oc, sa).",
    )
    parser.add_argument(
        "--ee-project",
        default=None,
        help="Google Earth Engine project ID (required by newer earthengine-api).",
    )
    args = parser.parse_args()

    run(args.master_csv, args.sword_cont_abb, ee_project=args.ee_project)

## Batch Processing: Fraction of Distance Downstream

Process all rivers for a single SWORD continent from a master CSV.
For each river the script:

1. Loads the reach shapefile and finds intersecting SWORD reaches.
2. Auto-selects the most downstream SWORD reach as the anchor.
3. Traces the full mainstem (upstream + downstream).
4. Samples DEM elevation at each traced reach midpoint via GEE.
5. Computes `frac_dist_ds` and `frac_dr_ar` per `ds_order` reach.
6. Saves a per-river `.gpkg` and a continent-level results CSV.
7. Generates a PDF report with map + inset + profile per reach.

Set the `BATCH_CONFIG` below, then run the cell.

In [1]:
# ── BATCH CONFIG ───────────────────────────────────────────────────
BATCH_CONFIG = {
    "master_csv": r"E:\Dissertation\Data\GreenbergZhaoRees_river_datasheet.csv",
    "sword_cont_abb": "sa",           # af, as, eu, na, oc, sa
    "sword_dir": r"E:\Dissertation\Data\SWORD",
    "sword_version": "v17b",
    "output_dir": r"E:\Dissertation\Data\SWORD\Extracted_Rivers\Chapter_2_dist_ds_analysis",
    "ee_project": None,               # set to "your-project-id" if needed
}
# ───────────────────────────────────────────────────────────────────

import os
import time as _time

import ee
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

try:
    import contextily as ctx
    _HAS_CTX = True
except ImportError:
    _HAS_CTX = False

ARCTICDEM_MIN_LAT = 60.0
VALID_CONTINENTS = {"af", "as", "eu", "na", "oc", "sa"}


# ── helpers ────────────────────────────────────────────────────────

def _get_elev(lat, lon, max_retries=3):
    for attempt in range(max_retries):
        try:
            pt = ee.Geometry.Point([lon, lat])
            if lat >= ARCTICDEM_MIN_LAT:
                dem = ee.Image("UMN/PGC/ArcticDEM/V3/2m_mosaic"); scale = 32
            else:
                dem = ee.Image("NASA/NASADEM_HGT/001"); scale = 30
            s = dem.select("elevation").sample(pt, scale).first()
            if s is None:
                return None
            v = s.get("elevation").getInfo()
            if v is None or not (-500 < v < 9000):
                return None
            return float(v)
        except Exception:
            if attempt < max_retries - 1:
                _time.sleep(2 ** attempt)
            else:
                return None
    return None


def _build_lookups(sw):
    dn = sw.set_index("reach_id")["rch_id_dn"].to_dict()
    dist = sw.set_index("reach_id")["dist_out"].to_dict()
    facc = sw.set_index("reach_id")["facc"].to_dict()
    up = {}
    for _, r in sw.iterrows():
        rid = int(r["reach_id"])
        ids = [int(x) for x in str(r["rch_id_dn"]).split() if int(x) != 0]
        for d in ids:
            up.setdefault(d, []).append(rid)
    return dn, dist, facc, up


def _trace_us(rid, up_lk, facc_lk, mx=5000):
    path = []
    cur = rid
    for _ in range(mx):
        cands = up_lk.get(cur, [])
        if not cands:
            break
        best = max(cands, key=lambda r: facc_lk.get(r, 0))
        path.append(best)
        cur = best
    return list(reversed(path))


def _trace_ds(rid, dn_lk, mx=5000):
    path = []
    cur = rid
    for _ in range(mx):
        raw = dn_lk.get(cur)
        if raw is None:
            break
        ids = [int(x) for x in str(raw).split() if int(x) != 0]
        if not ids:
            break
        cur = ids[0]
        path.append(cur)
    return path


def _sample_elevations(reaches_gdf):
    recs = []
    n = len(reaches_gdf)
    for i, (_, r) in enumerate(reaches_gdf.iterrows()):
        mid = r.geometry.interpolate(0.5, normalized=True)
        e = _get_elev(mid.y, mid.x)
        recs.append({"reach_id": int(r["reach_id"]),
                      "reach_order": int(r["reach_order"]),
                      "dist_out": float(r["dist_out"]),
                      "elevation_m": e})
        if (i + 1) % 25 == 0 or i == n - 1:
            ok = sum(1 for x in recs if x["elevation_m"] is not None)
            print(f"      ... {i+1}/{n} sampled ({ok} OK)")
    return pd.DataFrame(recs)


def _add_basemap(ax):
    if not _HAS_CTX:
        return
    try:
        ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
    except Exception:
        pass


def _draw_page(pdf, rname, dso, riv3857, ds_bnd3857, edf, srid, fdd, fda):
    fig = plt.figure(figsize=(16, 13))
    gs = fig.add_gridspec(2, 2, height_ratios=[3, 1], width_ratios=[2, 1])
    ax_map = fig.add_subplot(gs[0, 0])
    ax_ctx = fig.add_subplot(gs[0, 1])
    ax_prof = fig.add_subplot(gs[1, :])

    sel = riv3857[riv3857["reach_id"] == srid] if srid is not None else riv3857.iloc[:0]

    # map
    riv3857.plot(ax=ax_map, column="reach_order", cmap="viridis", linewidth=2.5,
                 legend=True, legend_kwds={"label": "Reach order (US \u2192 DS)", "shrink": 0.5})
    if not sel.empty:
        sel.plot(ax=ax_map, color="red", linewidth=4)
    ds_bnd3857.plot(ax=ax_map, facecolor="none", edgecolor="red", linewidth=2, linestyle="--")
    _add_basemap(ax_map)

    handles = [Line2D([0], [0], color="red", lw=2, ls="--", label=f"Reach {dso} boundary")]
    if srid is not None:
        handles.insert(0, Line2D([0], [0], color="red", lw=4, label=f"SWORD {srid}"))
    ax_map.legend(handles=handles, loc="best")
    if fdd is not None and fda is not None:
        ax_map.set_title(f"{rname} \u2014 Reach {dso}\nfrac_dist_ds = {fdd:.4f}   frac_dr_ar = {fda:.4f}")
    else:
        ax_map.set_title(f"{rname} \u2014 Reach {dso}\n(metrics unavailable)")
    ax_map.set_xlabel("Easting (m)")
    ax_map.set_ylabel("Northing (m)")

    # context inset
    xmin, xmax = ax_map.get_xlim()
    ymin, ymax = ax_map.get_ylim()
    xc, yc = (xmin+xmax)/2, (ymin+ymax)/2
    hw, hh = (xmax-xmin)*2, (ymax-ymin)*2
    riv3857.plot(ax=ax_ctx, color="gold", linewidth=1.2)
    if not sel.empty:
        sel.plot(ax=ax_ctx, color="red", linewidth=2.5)
    ax_ctx.set_xlim(xc-hw, xc+hw)
    ax_ctx.set_ylim(yc-hh, yc+hh)
    ax_ctx.set_aspect("equal")
    _add_basemap(ax_ctx)
    ax_ctx.add_patch(Rectangle((xmin, ymin), xmax-xmin, ymax-ymin,
                                fill=False, edgecolor="yellow", linewidth=1.5))
    ax_ctx.set_title("Regional context")
    ax_ctx.set_xticks([]); ax_ctx.set_yticks([])

    # profile
    vdf = edf.dropna(subset=["elevation_m"]).copy()
    if not vdf.empty:
        mx_do = vdf["dist_out"].max()
        vdf["dist_km"] = (mx_do - vdf["dist_out"]) / 1000
        ax_prof.plot(vdf["dist_km"], vdf["elevation_m"], "o-", color="steelblue",
                     markersize=3, linewidth=1.2, label="SWORD reaches")
        if srid is not None:
            s = vdf[vdf["reach_id"] == srid]
            if not s.empty:
                ax_prof.plot(s["dist_km"], s["elevation_m"], "ro", markersize=10,
                             zorder=5, label=f"Reach {dso}")
        ax_prof.legend(loc="best")
    ax_prof.set_title("Longitudinal profile")
    ax_prof.set_xlabel("Distance from headwaters (km)")
    ax_prof.set_ylabel("Elevation (m)")
    ax_prof.grid(True, alpha=0.3)

    plt.tight_layout()
    pdf.savefig(fig)
    plt.close(fig)


def _process_river(rname, wdir, sw, dn_lk, dist_lk, facc_lk, up_lk, out_dir):
    rpath = os.path.join(wdir, "RiverMapping", "Reaches", rname, f"{rname}.shp")
    if not os.path.isfile(rpath):
        print(f"    ERROR: not found {rpath}")
        return None, None, None, None

    rgdf = gpd.read_file(rpath)
    if rgdf.crs != sw.crs:
        rgdf = rgdf.to_crs(sw.crs)

    isect = gpd.sjoin(sw, rgdf, how="inner", predicate="intersects"
                      ).drop(columns="index_right", errors="ignore")
    isect = isect.drop_duplicates(subset="reach_id")
    if isect.empty:
        print("    ERROR: no SWORD reaches intersect reach boundary")
        return None, None, None, None

    anchor = int(isect.loc[isect["dist_out"].idxmin(), "reach_id"])
    print(f"    Anchor: {anchor}  (dist_out={isect.loc[isect['dist_out'].idxmin(), 'dist_out']/1000:.0f} km)")

    us = _trace_us(anchor, up_lk, facc_lk)
    ds = _trace_ds(anchor, dn_lk)
    fp = us + [anchor] + ds
    print(f"    Traced: {len(us)} US + 1 + {len(ds)} DS = {len(fp)} reaches")

    riv = sw[sw["reach_id"].isin(fp)].copy()
    omap = {rid: i+1 for i, rid in enumerate(fp)}
    riv["reach_order"] = riv["reach_id"].map(omap)
    riv = riv.sort_values("reach_order").reset_index(drop=True)

    riv_dir = os.path.join(out_dir, rname)
    os.makedirs(riv_dir, exist_ok=True)
    gpkg = os.path.join(riv_dir, f"{rname}_SWORD_reaches.gpkg")
    riv.to_file(gpkg, driver="GPKG")
    print(f"    Saved: {gpkg}")

    print(f"    Sampling elevations ({len(riv)} reaches) ...")
    edf = _sample_elevations(riv)
    ok = edf["elevation_m"].notna().sum()
    print(f"    Elevation: {ok}/{len(edf)} successful")

    mx_do = riv["dist_out"].max()
    mn_do = riv["dist_out"].min()
    tot = mx_do - mn_do

    has_facc = "facc" in riv.columns
    if has_facc:
        mx_facc = float(riv["facc"].max())
    else:
        mx_facc = None

    rpolys = rgdf[["ds_order", "geometry"]].rename(columns={"ds_order": "dso_poly"})
    s2d = gpd.sjoin(riv, rpolys, how="inner", predicate="intersects"
                    ).drop(columns="index_right", errors="ignore")

    results = []
    for dso in sorted(rgdf["ds_order"].unique()):
        dso = int(dso)
        m = s2d[s2d["dso_poly"] == dso]
        if m.empty:
            print(f"    ds_order {dso}: no SWORD overlap")
            results.append({"river_name": rname, "ds_order": dso,
                            "frac_dist_ds": None, "frac_dr_ar": None,
                            "sword_reach_id": None})
            continue
        t = m.loc[m["dist_out"].idxmin()]
        fdd = (mx_do - float(t["dist_out"])) / tot if tot > 0 else None
        fda = float(t["facc"]) / mx_facc if has_facc and mx_facc and mx_facc > 0 else None
        results.append({"river_name": rname, "ds_order": dso,
                        "frac_dist_ds": fdd, "frac_dr_ar": fda,
                        "sword_reach_id": int(t["reach_id"])})
        ms = (f"frac_dist_ds={fdd:.4f}  frac_dr_ar={fda:.4f}"
              if fdd is not None and fda is not None else "metrics unavailable")
        print(f"    ds_order {dso}: SWORD {int(t['reach_id'])}  {ms}")

    return riv, rgdf, edf, results


# ── run ────────────────────────────────────────────────────────────
cont = BATCH_CONFIG["sword_cont_abb"].lower()
assert cont in VALID_CONTINENTS, f"Invalid continent: {cont}"

master = pd.read_csv(BATCH_CONFIG["master_csv"])
for c in ("river_name", "working_directory", "SWORD_cont_abb"):
    assert c in master.columns, f"Missing column: {c}"
master["SWORD_cont_abb"] = master["SWORD_cont_abb"].str.strip().str.lower()
rivers = master[master["SWORD_cont_abb"] == cont].reset_index(drop=True)
assert len(rivers) > 0, f"No rivers for continent '{cont}'"

print(f"Rivers to process ({cont.upper()}): {len(rivers)}")
for _, r in rivers.iterrows():
    print(f"  - {r['river_name']}")
print()

sw_path = os.path.join(BATCH_CONFIG["sword_dir"],
                        f"{cont}_sword_reaches_{BATCH_CONFIG['sword_version']}.gpkg")
print(f"Loading {os.path.basename(sw_path)} ...")
batch_sw = gpd.read_file(sw_path, engine="pyogrio")
print(f"Loaded {len(batch_sw):,} SWORD reaches.")
if "facc" not in batch_sw.columns:
    print("WARNING: 'facc' column not found; frac_dr_ar will be None.")
print()

print("Building topology lookups ...")
b_dn, b_dist, b_facc, b_up = _build_lookups(batch_sw)
print(f"Upstream lookup: {len(b_up):,} entries.\n")

print("Initialising GEE ...")
if BATCH_CONFIG["ee_project"]:
    ee.Initialize(project=BATCH_CONFIG["ee_project"])
else:
    ee.Initialize()
print("GEE ready.\n")

out_dir = BATCH_CONFIG["output_dir"]
os.makedirs(out_dir, exist_ok=True)

all_results = []
pdf_queue = []

for i, (_, row) in enumerate(rivers.iterrows()):
    rn = row["river_name"]
    wd = row["working_directory"]
    print("=" * 60)
    print(f"[{i+1}/{len(rivers)}]  {rn}")
    print("=" * 60)
    riv, rgdf, edf, res = _process_river(rn, wd, batch_sw, b_dn, b_dist, b_facc, b_up, out_dir)
    if res is not None:
        all_results.extend(res)
        pdf_queue.append((rn, riv, rgdf, edf, res))
    print()

if all_results:
    rdf = pd.DataFrame(all_results)
    csv_out = os.path.join(out_dir, f"{cont}_fraction_ds_results.csv")
    rdf.to_csv(csv_out, index=False)
    print(f"Results CSV: {csv_out}")
    print(rdf.to_string(index=False))
    print()

if pdf_queue:
    pdf_out = os.path.join(out_dir, f"{cont}_fraction_ds_report.pdf")
    print(f"Generating PDF: {pdf_out}")
    with PdfPages(pdf_out) as pdf:
        for rn, riv, rgdf, edf, res in pdf_queue:
            riv3857 = riv.to_crs(epsg=3857)
            rgdf3857 = rgdf.to_crs(epsg=3857)
            for r in res:
                dso = r["ds_order"]
                bnd = rgdf3857[rgdf3857["ds_order"] == dso]
                if bnd.empty:
                    continue
                _draw_page(pdf, rn, dso, riv3857, bnd, edf,
                           r.get("sword_reach_id"),
                           r.get("frac_dist_ds"), r.get("frac_dr_ar"))
                print(f"  Page: {rn}  ds_order={dso}")
    print(f"\nPDF complete: {pdf_out}")

print("\nDone.")

Rivers to process (SA): 45
  - Amazonas_Jatuarana
  - Amazonas_Tamshiyacu
  - Araguaia_Aruana
  - Araguaia_LuizAlves
  - Araguaia_SaoFelixDoAraguaia
  - Cuiaba_PortoDoAlegre
  - Demini_PostoAjuricaba
  - Guapore_Pimenteiras
  - Ica_IpirangaVelho
  - Itacuai_LadarioJusante
  - Jurua_EirunepeMontante
  - Jurua_SantosDumont
  - Jutai_PortoSeguro
  - Madeira_Humaita
  - MadreDeDios_CachuelaEsperanza
  - Magdalena_Calamar
  - Magdalena_PuertoBerrio
  - Mamore_Guajara-Mirim
  - Mamore_PuertoSiles
  - Manas_Mathanguri
  - Maranon_Borja
  - Maranon_SanRegis
  - Mortes_SantoAntonioDoLeverger
  - Napo_Bellavista
  - Napo_NvoRocafuerte
  - Paraguay_Asuncion
  - Parana_Corrientes
  - Pilcomayo_VillaMontes
  - Purus_Canutama
  - Purus_Labrea
  - Purus_SeringalDaCaridade
  - Purus_ValparaisoMontante
  - SaoFrancisco_BomJesusDaLapa
  - SaoFrancisco_Morpara
  - Solimoes_Itapeua
  - Solimoes_Manacapuru
  - Solimoes_SantoAntonioDoIca
  - Solimoes_SaoPauloDeOlivenca
  - Solimoes_Tabatinga
  - Tarauaca_En